# 39장 · 그래프 실행 기록 · Streamlit에서 근거와 종료 경로 읽기

Streamlit에서는 화면을 다시 그릴 때도 유지할 값을 st.session_state에 담습니다. 그래프를 실행하는 코드와 이전 결과를 표시하는 코드를 나누면 화면을 열 때마다 검색하지 않아도 됩니다.

## st.button으로 버튼을 누른 실행만 기록하기

`st.button()`은 버튼을 클릭해 앱이 다시 실행될 때 True를 반환합니다. 그때만 `st.session_state`에 한 건을 추가하면 다른 위젯을 건드려도 기존 건수가 유지됩니다. 아래 셀을 실행하면 현재 폴더에 작은 앱 파일이 생깁니다. 수업 환경에 Streamlit이 없으면 노트북에서 `%pip install streamlit`을 먼저 실행하세요.

In [ ]:
from pathlib import Path

app_code = '''
import streamlit as st

if "runs" not in st.session_state:
    st.session_state.runs = []
if st.button("기록 추가"):
    st.session_state.runs.append("완료")
st.checkbox("다른 이유로 다시 실행")
st.write("기록 수", len(st.session_state.runs))
'''.lstrip()
Path("button_example.py").write_text(app_code, encoding="utf-8")
print("button_example.py")

button_example.py


같은 Python 환경에서 터미널을 열고, 방금 파일을 만든 폴더에서 다음 명령을 실행하세요. 터미널에 표시된 Local URL을 브라우저에서 엽니다.

```bash
python -m streamlit run button_example.py
```

처음 보이는 기록 수는 0입니다. ‘기록 추가’를 누르고 체크박스를 바꾼 뒤 다시 ‘기록 추가’를 누르세요. 숫자가 차례로 1, 1, 2가 되면 버튼으로 보낸 입력만 목록에 들어간 것입니다. 실험이 끝나면 터미널에서 Ctrl+C로 앱을 종료하세요.

## 응용 실습 · 화면에서 그래프 경로 확인하기

짝 활동에는 [실습 수첩](pre-mid/documents/graph/workshop.md)을 사용합니다. 과제 아래에서 풀이를 펼칠 수 있습니다.

## 사용자가 고를 것과 자동으로 남길 것

질문자는 모델명과 사용 시점을 알려 줍니다. 수업용 화면은 이 정보를 보고 기종·판본을 직접 선택하게 합니다. 학생은 설명서 표지의 적용 기간을 확인합니다. 검색 횟수·검색어·방문 경로·근거 쪽은 프로그램이 남깁니다. 서로 다른 주제의 두 질문은 독립 검수 사례로 취급합니다.

3부에서 쓰던 세 PDF와 공용 검색기를 같은 폴더에서 엽니다. 교재 ZIP 안의 `pre-mid/documents/rag`, `graph`, `sources`가 함께 있어야 합니다. 실제 검색 셀은 3부 Chroma 색인·OpenAI 임베딩과 API 키를 사용합니다. 처음 실행한다면 노트북 셀에서 `%pip install "langgraph>=1,<2" "langchain-openai>=1,<2" "chromadb>=1,<2" pypdf openai python-dotenv streamlit`를 실행하고 커널을 다시 시작하세요. PDF 직접 읽기와 고정 입력 경로 실험은 키 없이 실행됩니다.

In [1]:
from pathlib import Path
import sys

# 통합 ZIP과 저장소에서 같은 자료 폴더를 찾습니다.
docs = next(p / "pre-mid" / "documents" for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pre-mid" / "documents" / "graph").is_dir())
sys.path.insert(0, str(docs / "graph"))
sys.path.insert(0, str(docs / "rag"))
from rag_lab import MANUALS, read_pages

# 세 설명서의 모든 페이지를 실제 PDF에서 읽습니다.
pages = [page for path in MANUALS for page in read_pages(path)]
by_id = {page["id"]: page for page in pages}
from rag_lab import index_version
for p in pages:
    print(p["id"], p["model"], p["edition"])
print("색인 버전", index_version(pages))

recorder-a-1.0.pdf#p1 A 1.0
recorder-a-1.0.pdf#p2 A 1.0
recorder-a-1.1.pdf#p1 A 1.1
recorder-a-1.1.pdf#p2 A 1.1
recorder-b-1.0.pdf#p1 B 1.0
recorder-b-1.0.pdf#p2 B 1.0
색인 버전 recorder-2bfb240a53a6e8ec


세 PDF의 기종과 판본이 다르고 출처 ID는 모두 여섯 개입니다. 이 기준 색인에 대한 결과임을 다음 팀에게 전달할 때 버전을 함께 적습니다.

## 답을 보기 전에 어떤 경로를 예상하는가

표시등 질문은 올바른 판본이 첫 후보에 있으면 직접 답할 수 있습니다. 일상어로 바꿔 써서 처음 놓쳤다면 재검색 후 같은 페이지를 찾을 수도 있습니다. 무선 전송처럼 근거가 없는 질문은 보류해야 합니다. 실제 검색기가 “빨간불”도 곧장 찾으면 그 결과를 그대로 남기세요. 특정 질문에 특정 경로가 반드시 발생한다고 약속할 수 없습니다. 경로 자체의 세 가지 동작은 38장의 고정 입력으로 검증했습니다.

## st.session_state에 실행 결과를 저장하고 다시 표시하기

`st.selectbox`에서 고른 값은 이번 실행의 제품 설정입니다. 질문을 제출했을 때 그 설정과 그래프 결과를 함께 저장하세요. 이전 결과를 그릴 때에는 저장한 설정을 읽어야, 나중에 다른 판본을 고르더라도 앞서 받은 답의 출처가 바뀌지 않습니다.

이 셀은 실제 실행할 화면 파일을 만듭니다. 실행 전에 3부 자료와 .env를 펼친 노트북 폴더에 두세요. 터미널에서 `python -m streamlit run chapter39_recorder.py`를 실행합니다. 선택값을 바꿀 때 스크립트는 다시 실행되지만, 실행 버튼 역할을 하는 채팅 입력에 새 질문이 있을 때만 그래프를 호출합니다.

`st.selectbox`에서 고른 “A / 1.1”은 문자열 하나입니다. `split(" / ")`이 이를 기종 “A”와 판본 “1.1” 두 인자로 나눕니다. `ask(graph, question, model, edition)`에서 `model`이 가리키는 것은 이 녹음기의 기종입니다. 답변용 LLM의 이름은 여기에 넣지 않습니다. 결과 딕셔너리를 세션에 남기므로 사용자는 재검색을 실행하지 않고도 지난 근거를 다시 펼칠 수 있습니다.

In [2]:
from pathlib import Path

# 터미널에서 실행할 화면 파일을 현재 폴더에 저장합니다.
app_source = '''from pathlib import Path
import sys
import streamlit as st

docs = next(p / "pre-mid" / "documents" for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pre-mid" / "documents" / "graph").is_dir())
sys.path.insert(0, str(docs / "rag"))
sys.path.insert(0, str(docs / "graph"))
from rag_lab import open_index
from recorder_graph import ask, make_graph

st.title("녹음기 설명서 문의")
st.caption("질문에 적힌 기종과 사용 시점에 맞는 설명서를 먼저 고르세요.")
manual = st.selectbox("제품과 판본", ["A / 1.1", "A / 1.0", "B / 1.0"])
model, edition = manual.split(" / ")
if "reviews" not in st.session_state:
    st.session_state.reviews = []
if st.button("이 화면의 기록 비우기"):
    st.session_state.reviews = []

if question := st.chat_input("기종·사용 시점·행동을 포함해 물어보세요"):
    # 같은 세션의 후속 질문은 색인을 다시 열지 않습니다.
    if "recorder_graph" not in st.session_state:
        st.session_state.recorder_graph = make_graph(open_index(), k=2)
    with st.spinner("설명서 확인 중"):
        result = ask(st.session_state.recorder_graph, question, model, edition)
    st.session_state.reviews.append(result)

for item in st.session_state.reviews:
    st.chat_message("user").write(item["question"])
    with st.chat_message("assistant"):
        st.write(item["answer"])
        st.caption(f'{item["model"]}형 {item["edition"]}판 · 검색 {item["attempt"]}회 · {item["status"]}')
        for cite in item["citations"]:
            st.write(f'{cite["source"]} {cite["page"]}쪽 — {cite["quote"]}')
        with st.expander("검색 문단과 종료 이유"):
            for row in item["searches"]:
                st.write(row["query"])
                for page in row["documents"]:
                    st.write(page["id"], page["text"])
            st.write(" → ".join(item["path"]))
            st.write(item["reason"])
'''
Path("chapter39_recorder.py").write_text(app_source, encoding="utf-8")
print("chapter39_recorder.py")

chapter39_recorder.py


이 셀의 출력은 생성한 파일 이름입니다. 화면과 실제 모델 응답은 터미널에서 앱을 실행한 뒤 확인합니다.

## 화면에서 세 번 확인하기

A형 1.1판을 고르고 “4월에 화면을 어둡게 하면 녹음 중 빨간불도 꺼지나요?”를 보냅니다. 답의 인용을 눌러 보듯 실제 A형 1.1판 PDF 1쪽을 열어 그 문장을 찾으세요. 그다음 선택만 B형으로 바꾸고 이전 응답에 A형 1.1판 표기가 남는지 봅니다. 마지막으로 B형의 무선 전송을 물어 실제로 답했는지 보류했는지 기록하세요. 화면 공통 선택과 이미 끝난 실행의 조건을 혼동하면 예전 답이 다른 기종의 답처럼 보입니다.

새 질문을 보낼 때만 reviews 길이가 하나 늘어나야 합니다. 원문 펼치기, 선택값 변경, 버튼 없는 화면 다시 그리기는 새로운 답을 추가할 이유가 없습니다. 근거가 있는데도 보류한 경우와 근거가 없는데도 답한 경우를 서로 다른 문제로 기록합니다.

## 실습 · 짝에게 숨겨 둔 질문으로 검수하기

한 사람은 PDF와 질문만 보고 원문에서 기대 답·쪽을 적습니다. 다른 사람은 기대 답을 보지 않고 화면을 실행합니다. 두 결과가 다르면 검색 문단과 실제 인용을 펼쳐 처음 어긋난 곳을 찾습니다. 아래 셀에는 본인 질문·판본·기대 근거를 적어 원문이 맞는지 대조하세요.

In [3]:
question = "4월 A형으로 46분 녹음하면 중간에 녹음이 끊기나요?"
# by_id에서 기대 쪽을 찾으세요. 모델 답과 원문의 파일 분할 간격, 녹음 지속 여부를 각각 비교하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>question = &quot;4월 A형으로 46분 녹음하면 중간에 녹음이 끊기나요?&quot;
expected = by_id[&quot;recorder-a-1.1.pdf#p2&quot;]
print(question, expected[&quot;source&quot;], expected[&quot;page&quot;])
print(&quot;파일이 바뀌어도 녹음은 계속됩니다.&quot; in &quot; &quot;.join(expected[&quot;text&quot;].split()))</code></pre><p>예시 기대 답은 45분 뒤 새 파일이 생겨도 녹음은 계속된다는 것입니다. 실제 모델 답안에는 같은 뜻이 다른 말로 나올 수 있습니다. 45분과 계속 녹음이라는 두 조건을 따로 확인하세요.</p></details>

## 수업에서 보관한 두 실행을 원문과 대조하기

API를 다시 부르기 전에 기존 실행을 읽어 보세요. 제공한 JSON에는 6부에서 같은 설명서로 측정한 질문별 결과가 있습니다. D01은 A형의 녹음 중 표시등, D05는 B형의 무선 전송 방법을 묻습니다. 두 실행에서 마지막 노드와 인용 목록이 어떻게 달라야 하는지 예상하세요. 이 셀은 이미 저장된 기록만 엽니다.

In [4]:
import json

saved = json.loads((docs / "evaluation" / "dev-observed.json").read_text(encoding="utf-8"))
for run in saved["rows"]:
    if run["version"] == "graph-k2" and run["id"] in {"D01", "D05"}:
        output = run["output"]
        print(run["id"], " → ".join(output["path"]))
        print(output["answer"], output["citations"])

D01 retrieve → judge → compose
아니요, 녹음 중 앞쪽 빨간불을 안 보이게 할 수 없습니다. ['recorder-a-1.1.pdf#p1']
D05 retrieve → judge → rewrite → retrieve → judge → hold
선택한 설명서에서 답을 확인할 근거가 부족합니다. []


저장된 D01은 첫 검색 후 A형 1.1판 1쪽을 인용했습니다. D05는 다시 검색한 뒤에도 무선 전송 근거를 확인하지 못해 빈 인용 목록으로 끝났습니다. D05의 케이블 복사 설명을 찾아보며 무선 전송이 불가능하다는 결론과 어떻게 다른지 짝에게 설명해 보세요. 지금 화면에서 새로 실행한 결과가 다르면 두 실행을 구별해 기록합니다.

## 세 건의 검수 메모 중 어느 것이 재현 가능한가

1. “두 번째에야 맞았다.”
2. “A형 표시등 질문이 틀렸다. 모델을 바꾸자.”
3. “4월 A형에서 표시등을 숨기는지 물었다. 첫 후보 둘은 B형이었다. 재검색 뒤 A형 1.1판 1쪽이 들어왔고 화면은 그 쪽을 인용했다.”

첫 메모에는 무엇을 다시 입력할지, 둘째 메모에는 어디서 틀렸는지가 없습니다. 셋째 메모에는 입력 조건과 후보 변화가 있지만 아직 실제 질문 문장, 색인 버전, 검색어가 빠져 있습니다. 짝에게 셋째 메모만 준 뒤 똑같이 실행해 보게 하세요. 서로 다른 문구로 질문하면 첫 검색 순서가 달라질 수 있습니다. 이 차이가 다음 절에서 질문과 검색 기록을 함께 저장하는 이유입니다.

## 평가에서 같은 실행을 찾을 수 있도록 기록하기

팀원에게 “답이 틀렸다”만 남기면 어디서 다시 시작할지 모릅니다. 질문 원문, 선택 기종·판본, 색인 버전과 k, 각 검색어와 후보 ID, 최종 인용, 종료 이유, 사람이 기대한 쪽을 함께 건넵니다. 모델 응답 전체에 개인 정보가 들어간 실제 문의 대신 이 수업의 가상 제품 문서와 질문을 사용하세요.

In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# 8장에서 준비한 .env를 읽습니다. 키는 출력하지 않습니다.
load_dotenv()
use_live = bool(os.getenv("OPENAI_API_KEY"))
model_name = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
if use_live:
    client = OpenAI()

In [6]:
if use_live:
    import json
    from rag_lab import open_index
    from recorder_graph import ask, make_graph
    question = "4월 A형으로 46분 녹음하면 중간에 녹음이 끊기나요?"
    run = ask(make_graph(open_index(), k=2), question, "A", "1.1")
    transfer = {"question": question, "model": run["model"], "edition": run["edition"],
                "index_version": index_version(pages), "k": 2, "expected_page": "recorder-a-1.1.pdf#p2",
                "searches": [{"query": x["query"], "ids": [p["id"] for p in x["documents"]]}
                             for x in run["searches"]], "path": run["path"], "citations": run["citations"],
                "answer": run["answer"], "status": run["status"], "reason": run["reason"]}
    Path("chapter39-handoff.json").write_text(json.dumps(transfer, ensure_ascii=False, indent=2), encoding="utf-8")
    print(transfer["path"], transfer["status"], transfer["citations"])
else:
    print("실제 인계 파일은 API 키로 실행한 결과에서 만듭니다.")

실제 인계 파일은 API 키로 실행한 결과에서 만듭니다.


생성한 파일은 자신의 실제 실행 기록입니다. 40장에서는 같은 질문으로 LangSmith 기록을 열고, 검색 후보와 최종 인용 중 어디가 달라졌는지 찾아갑니다.

## 실습 · 설정 변경을 채택할지 정하기

짝은 후보 수 k를 2에서 4로 바꾸자고 합니다. 누락됐던 정답 쪽 하나를 찾은 경우와 무관한 쪽만 더해진 경우를 나눠 생각하세요. 같은 질문과 색인으로 한 조건만 바꾼 실행을 준비하고, 결과를 채택하려면 어떤 증거가 더 필요한지 적으세요.

In [7]:
limits = [2, 4]
# 같은 질문과 색인으로 k만 바꾸세요. 첫 검색의 문서 ID와 최종 citations에 기대 쪽이 있는지 각각 확인하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>if use_live:
    shared_index = open_index()
    for limit in (2, 4):
        r = ask(make_graph(shared_index, k=limit), question, &quot;A&quot;, &quot;1.1&quot;)
        print(limit, [p[&quot;id&quot;] for p in r[&quot;searches&quot;][0][&quot;documents&quot;]], r[&quot;status&quot;], r[&quot;citations&quot;])</code></pre><p>검색 후보 안에 기대 쪽이 들어왔는지, 최종 답이 그 쪽을 인용했는지 따로 봅니다. 한 질문에서 이겼다면 기존에 맞던 질문도 그대로 맞는지 다음 장에서 검사합니다.</p></details>

## 전이 과제 · 다른 판본으로 바꿨더니 옛 답의 제목이 바뀐 경우

사용자가 A형 1.1판으로 물어본 뒤 드롭다운에서 B형을 선택합니다. 화면은 위에서부터 다시 실행됩니다. 이전 기록을 그릴 때 현재 드롭다운 값으로 출처 머리글을 만들면, 어제의 A형 답이 갑자기 B형 결과로 표시됩니다. 검색이 잘못된 것처럼 보이지만 실제로는 표시할 때 읽은 값이 다릅니다.

새 질문을 보내기 전과 보낸 후를 구분해 테스트합니다. A형 질문 1회, 판본만 변경, B형 질문 1회, 초기화 차례로 조작하세요. 각 시점에 보여야 할 질문 수와 각 답의 기종을 종이에 먼저 적고 실제 화면과 맞춥니다. 어긋나면 선택 위젯의 현재 값과 각 결과 딕셔너리에 저장된 값을 따로 봅니다.

In [8]:
history = [{"model": "A", "edition": "1.1", "question": "첫 문의"}]
selected_model = "B"
# 새로 선택한 기종과 history[0]에 저장된 기종·판본을 읽으세요. 첫 답의 제목에 어떤 값을 써야 하나요?

<details><summary>모범 예시와 풀이</summary><pre><code>history = [{&quot;model&quot;: &quot;A&quot;, &quot;edition&quot;: &quot;1.1&quot;, &quot;question&quot;: &quot;첫 문의&quot;}]
selected_model = &quot;B&quot;
print(&quot;다음 질문에 쓸 기종&quot;, selected_model)
print(&quot;첫 답에 표시할 기종&quot;, history[0][&quot;model&quot;], history[0][&quot;edition&quot;])</code></pre><p>보이는 기록 수는 1, 1, 2, 0이어야 합니다. 첫 답은 새 입력을 받기 전후 모두 A형 1.1판입니다. 이전 결과를 그릴 때는 각 item의 model과 edition을 사용합니다. 이 검사는 모델 호출 없이도 재현할 수 있습니다.</p></details>

[Streamlit 채팅 입력](https://docs.streamlit.io/develop/api-reference/chat/st.chat_input) · [LangGraph 실행 기록](https://docs.langchain.com/oss/python/langgraph/persistence)